# Round 4 — Market-Maker Deep Dive: Mark 14 / 38 / 55

**Key structure discovered:**
- **Mark 14** — market-maker in HP, VEX, VEV_4000. Buys at bid, sells at ask. Captures the spread.
- **Mark 38** — pays Mark 14's spread in HP and VEV_4000. Directional/informed trader.
- **Mark 55** — intermediary in VEX: pays Mark 14 the spread, then resells to Mark 01/67.

Mark 38 and Mark 55 **never trade with each other**. Each has a fixed relationship with Mark 14.

## 1 — Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

DATA_DIR = Path("../../historical_data/round_4")

# ── trades ──
trade_frames = []
for d in [1, 2, 3]:
    tmp = pd.read_csv(DATA_DIR / f"trades_round_4_day_{d}.csv", sep=";")
    tmp["day"] = d
    trade_frames.append(tmp)
trades = pd.concat(trade_frames, ignore_index=True)
trades["global_ts"] = (trades["day"] - 1) * 1_000_000 + trades["timestamp"]

# ── prices (for mid-price reference) ──
price_frames = []
for d in [1, 2, 3]:
    tmp = pd.read_csv(DATA_DIR / f"prices_round_4_day_{d}.csv", sep=";")
    price_frames.append(tmp)
prices = pd.concat(price_frames, ignore_index=True)
prices["global_ts"] = (prices["day"] - 1) * 1_000_000 + prices["timestamp"]

MMs      = ["Mark 14", "Mark 38", "Mark 55"]
MM_PRODS = {
    "Mark 14": ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT", "VEV_4000"],
    "Mark 38": ["HYDROGEL_PACK", "VEV_4000"],
    "Mark 55": ["VELVETFRUIT_EXTRACT"],
}
days = [1, 2, 3]
day_colors = {1: "#4C72B0", 2: "#DD8452", 3: "#55A868"}
ticks_per_day = 1_000_000

# ── per-trader activity in long form ──
def trader_activity(trader):
    t = trades[(trades["buyer"] == trader) | (trades["seller"] == trader)].copy()
    t["side"]       = t.apply(lambda r: "buy" if r["buyer"] == trader else "sell", axis=1)
    t["signed_qty"] = t.apply(lambda r: r["quantity"] if r["buyer"] == trader else -r["quantity"], axis=1)
    t["counterparty"] = t.apply(lambda r: r["seller"] if r["buyer"] == trader else r["buyer"], axis=1)
    return t.sort_values("global_ts").reset_index(drop=True)

act = {mm: trader_activity(mm) for mm in MMs}
print("Loaded.")
for mm in MMs:
    a = act[mm]
    print(f"  {mm}: {len(a)} trades across {a['symbol'].nunique()} products")

## 2 — Counterparty Relationship Map

In [ ]:
for mm in MMs:
    print(f"\n{mm} — who they trade with per product:")
    pivot = act[mm].groupby(["symbol", "counterparty", "side"])["quantity"].sum().unstack(fill_value=0)
    print(pivot.to_string())

## 3 — Inventory Over Time

Cumulative signed position per product. Market-makers should mean-revert toward zero.

In [ ]:
def build_inventory(trader, product):
    t = act[trader]
    t = t[t["symbol"] == product].copy()
    t = t.sort_values("global_ts").reset_index(drop=True)
    t["inventory"] = t["signed_qty"].cumsum()
    return t


# --- HYDROGEL_PACK: Mark 14 vs Mark 38 ---
inv14_hp = build_inventory("Mark 14", "HYDROGEL_PACK")
inv38_hp = build_inventory("Mark 38", "HYDROGEL_PACK")

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
for d in days:
    sub14 = inv14_hp[inv14_hp["day"] == d]
    sub38 = inv38_hp[inv38_hp["day"] == d]
    axes[0].step(sub14["global_ts"], sub14["inventory"], color=day_colors[d], lw=1.2,
                 label=f"Mark 14 day {d}", where="post")
    axes[1].step(sub38["global_ts"], sub38["inventory"], color=day_colors[d], lw=1.2,
                 label=f"Mark 38 day {d}", where="post")
for ax in axes:
    ax.axhline(0, color="black", lw=0.7, ls="--")
    for i in range(1, len(days)):
        ax.axvline(i * ticks_per_day, color="grey", ls="--", lw=0.6, alpha=0.5)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_title("HYDROGEL_PACK — Mark 14 Inventory", fontsize=11)
axes[1].set_title("HYDROGEL_PACK — Mark 38 Inventory", fontsize=11)
axes[1].set_xlabel("Global timestamp")
for ax in axes:
    ax.set_ylabel("Net position")
plt.tight_layout()
plt.show()

# --- VELVETFRUIT_EXTRACT: Mark 14 vs Mark 55 ---
inv14_vex = build_inventory("Mark 14", "VELVETFRUIT_EXTRACT")
inv55_vex = build_inventory("Mark 55", "VELVETFRUIT_EXTRACT")

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
for d in days:
    sub14 = inv14_vex[inv14_vex["day"] == d]
    sub55 = inv55_vex[inv55_vex["day"] == d]
    axes[0].step(sub14["global_ts"], sub14["inventory"], color=day_colors[d], lw=1.2,
                 label=f"Mark 14 day {d}", where="post")
    axes[1].step(sub55["global_ts"], sub55["inventory"], color=day_colors[d], lw=1.2,
                 label=f"Mark 55 day {d}", where="post")
for ax in axes:
    ax.axhline(0, color="black", lw=0.7, ls="--")
    for i in range(1, len(days)):
        ax.axvline(i * ticks_per_day, color="grey", ls="--", lw=0.6, alpha=0.5)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_title("VELVETFRUIT_EXTRACT — Mark 14 Inventory", fontsize=11)
axes[1].set_title("VELVETFRUIT_EXTRACT — Mark 55 Inventory", fontsize=11)
axes[1].set_xlabel("Global timestamp")
for ax in axes:
    ax.set_ylabel("Net position")
plt.tight_layout()
plt.show()

# --- VEV_4000: Mark 14 vs Mark 38 ---
inv14_vev = build_inventory("Mark 14", "VEV_4000")
inv38_vev = build_inventory("Mark 38", "VEV_4000")

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
for d in days:
    sub14 = inv14_vev[inv14_vev["day"] == d]
    sub38 = inv38_vev[inv38_vev["day"] == d]
    axes[0].step(sub14["global_ts"], sub14["inventory"], color=day_colors[d], lw=1.2,
                 label=f"Mark 14 day {d}", where="post")
    axes[1].step(sub38["global_ts"], sub38["inventory"], color=day_colors[d], lw=1.2,
                 label=f"Mark 38 day {d}", where="post")
for ax in axes:
    ax.axhline(0, color="black", lw=0.7, ls="--")
    for i in range(1, len(days)):
        ax.axvline(i * ticks_per_day, color="grey", ls="--", lw=0.6, alpha=0.5)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_title("VEV_4000 — Mark 14 Inventory", fontsize=11)
axes[1].set_title("VEV_4000 — Mark 38 Inventory", fontsize=11)
axes[1].set_xlabel("Global timestamp")
for ax in axes:
    ax.set_ylabel("Net position")
plt.tight_layout()
plt.show()

## 4 — Inventory Statistics & Mean-Reversion

In [ ]:
pairs = [
    ("Mark 14", "HYDROGEL_PACK"),
    ("Mark 38", "HYDROGEL_PACK"),
    ("Mark 14", "VELVETFRUIT_EXTRACT"),
    ("Mark 55", "VELVETFRUIT_EXTRACT"),
    ("Mark 14", "VEV_4000"),
    ("Mark 38", "VEV_4000"),
]
rows = []
for trader, product in pairs:
    inv = build_inventory(trader, product)
    rows.append({
        "trader":    trader,
        "product":   product,
        "min":       inv["inventory"].min(),
        "max":       inv["inventory"].max(),
        "mean":      inv["inventory"].mean().round(1),
        "std":       inv["inventory"].std().round(1),
        "end_day1":  inv[inv["day"]==1]["inventory"].iloc[-1] if len(inv[inv["day"]==1]) else 0,
        "end_day2":  inv[inv["day"]==2]["inventory"].iloc[-1] if len(inv[inv["day"]==2]) else 0,
        "end_day3":  inv[inv["day"]==3]["inventory"].iloc[-1] if len(inv[inv["day"]==3]) else 0,
    })
stats_df = pd.DataFrame(rows).set_index(["trader", "product"])
print("Inventory stats (by end-of-day position and range):")
print(stats_df.to_string())

## 5 — Spread Analysis: Buy Price vs Sell Price Over Time

In [ ]:
def spread_plot(trader, product, window=20):
    t = act[trader][act[trader]["symbol"] == product].copy().sort_values("global_ts")
    buy_t  = t[t["side"] == "buy"].copy()
    sell_t = t[t["side"] == "sell"].copy()

    # rolling average buy/sell price
    buy_t["roll_price"]  = buy_t["price"].rolling(window, min_periods=1).mean()
    sell_t["roll_price"] = sell_t["price"].rolling(window, min_periods=1).mean()

    # mid-price from prices df
    mid = prices[prices["product"] == product][["global_ts", "mid_price"]].sort_values("global_ts")

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

    ax1.plot(mid["global_ts"], mid["mid_price"], color="grey", lw=0.6, alpha=0.7, label="Mid price")
    ax1.scatter(buy_t["global_ts"],  buy_t["price"],  s=8, color="#55A868", alpha=0.5, label="Buy trades",  zorder=3)
    ax1.scatter(sell_t["global_ts"], sell_t["price"], s=8, color="#C44E52", alpha=0.5, label="Sell trades", zorder=3)
    ax1.plot(buy_t["global_ts"],  buy_t["roll_price"],  color="#55A868", lw=1.5, label=f"Roll buy ({window})",  zorder=4)
    ax1.plot(sell_t["global_ts"], sell_t["roll_price"], color="#C44E52", lw=1.5, label=f"Roll sell ({window})", zorder=4)
    ax1.legend(fontsize=8)
    ax1.set_title(f"{trader} | {product} — Buy/Sell Prices vs Mid", fontsize=11)
    ax1.set_ylabel("Price")
    ax1.grid(True, alpha=0.3)

    # instantaneous spread at each buy/sell pair (rolling)
    # interpolate roll_price onto common ts
    all_ts = sorted(set(buy_t["global_ts"]).union(sell_t["global_ts"]))
    roll_buy  = np.interp(all_ts, buy_t["global_ts"],  buy_t["roll_price"])
    roll_sell = np.interp(all_ts, sell_t["global_ts"], sell_t["roll_price"])
    ax2.plot(all_ts, roll_sell - roll_buy, color="#4C72B0", lw=1.2, label="Roll sell – Roll buy")
    ax2.axhline(0, color="black", lw=0.6, ls="--")
    ax2.fill_between(all_ts, roll_sell - roll_buy, 0,
                     where=(np.array(roll_sell) > np.array(roll_buy)),
                     color="#55A868", alpha=0.2, label="Positive (MM earns)")
    ax2.fill_between(all_ts, roll_sell - roll_buy, 0,
                     where=(np.array(roll_sell) < np.array(roll_buy)),
                     color="#C44E52", alpha=0.2, label="Negative (taker pays)")
    ax2.legend(fontsize=8)
    ax2.set_title(f"Rolling Spread Captured (sell price − buy price)", fontsize=10)
    ax2.set_ylabel("Spread")
    ax2.set_xlabel("Global timestamp")
    ax2.grid(True, alpha=0.3)
    for ax in (ax1, ax2):
        for i in range(1, len(days)):
            ax.axvline(i * ticks_per_day, color="grey", ls="--", lw=0.6, alpha=0.5)
    plt.tight_layout()
    plt.show()


spread_plot("Mark 14", "HYDROGEL_PACK")
spread_plot("Mark 38", "HYDROGEL_PACK")
spread_plot("Mark 14", "VELVETFRUIT_EXTRACT")
spread_plot("Mark 55", "VELVETFRUIT_EXTRACT")
spread_plot("Mark 14", "VEV_4000")
spread_plot("Mark 38", "VEV_4000")

## 6 — Price Paid vs Mid Price (Trade Premium / Discount)

In [ ]:
def price_vs_mid(trader, product):
    """How far above/below mid does this trader buy and sell?"""
    t = act[trader][act[trader]["symbol"] == product].copy()
    mid = prices[prices["product"] == product][["global_ts", "mid_price"]].sort_values("global_ts")

    # merge nearest mid_price by timestamp (merge_asof)
    t = t.sort_values("global_ts")
    mid = mid.sort_values("global_ts")
    merged = pd.merge_asof(t, mid, on="global_ts")
    merged["premium"] = merged["price"] - merged["mid_price"]
    return merged


plot_pairs = [
    ("Mark 14", "HYDROGEL_PACK"),   ("Mark 38", "HYDROGEL_PACK"),
    ("Mark 14", "VELVETFRUIT_EXTRACT"), ("Mark 55", "VELVETFRUIT_EXTRACT"),
    ("Mark 14", "VEV_4000"),         ("Mark 38", "VEV_4000"),
]

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for ax, (trader, product) in zip(axes, plot_pairs):
    m = price_vs_mid(trader, product)
    buy_m  = m[m["side"] == "buy"]
    sell_m = m[m["side"] == "sell"]

    ax.axhline(0, color="black", lw=0.8, ls="--", alpha=0.6)
    ax.scatter(buy_m["global_ts"],  buy_m["premium"],  s=6,  color="#55A868", alpha=0.5, label="Buy premium")
    ax.scatter(sell_m["global_ts"], sell_m["premium"], s=6,  color="#C44E52", alpha=0.5, label="Sell premium")

    # rolling means
    if len(buy_m) > 5:
        buy_roll = buy_m.copy()
        buy_roll["roll"] = buy_roll["premium"].rolling(20, min_periods=1).mean()
        ax.plot(buy_roll["global_ts"], buy_roll["roll"], color="#55A868", lw=1.5)
    if len(sell_m) > 5:
        sell_roll = sell_m.copy()
        sell_roll["roll"] = sell_roll["premium"].rolling(20, min_periods=1).mean()
        ax.plot(sell_roll["global_ts"], sell_roll["roll"], color="#C44E52", lw=1.5)

    mean_buy_prem  = buy_m["premium"].mean()  if len(buy_m)  > 0 else float("nan")
    mean_sell_prem = sell_m["premium"].mean() if len(sell_m) > 0 else float("nan")
    ax.set_title(f"{trader} | {product}\nbuy Δmid={mean_buy_prem:.1f}  sell Δmid={mean_sell_prem:.1f}",
                 fontsize=9)
    ax.set_ylabel("Price − Mid")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    for i in range(1, len(days)):
        ax.axvline(i * ticks_per_day, color="grey", ls="--", lw=0.6, alpha=0.4)

fig.suptitle("Trade Price vs Mid Price — Premium/Discount per Trader", fontsize=13)
plt.tight_layout()
plt.show()

## 7 — Mark 38: Is They Informed? Price Impact After Trades

In [ ]:
"""If Mark 38 pays the spread but is an informed trader, the price should move
in their direction after their trade. We measure mid-price change N ticks after
each Mark 38 trade."""

def price_impact(trader, product, horizons=[500, 1000, 2000, 5000, 10000]):
    t = act[trader][act[trader]["symbol"] == product].copy().sort_values("global_ts")
    mid = prices[prices["product"] == product][["global_ts", "mid_price"]].sort_values("global_ts").reset_index(drop=True)

    rows = []
    for _, trade in t.iterrows():
        # mid at trade time
        idx0 = mid["global_ts"].searchsorted(trade["global_ts"])
        if idx0 >= len(mid):
            continue
        mid0 = mid.iloc[idx0]["mid_price"]
        row = {"ts": trade["global_ts"], "side": trade["side"], "price": trade["price"]}
        for h in horizons:
            idx_h = mid["global_ts"].searchsorted(trade["global_ts"] + h)
            if idx_h < len(mid):
                mid_h = mid.iloc[idx_h]["mid_price"]
                # signed change: +ve means price moved in buy direction
                sign = 1 if trade["side"] == "buy" else -1
                row[f"impact_{h}"] = sign * (mid_h - mid0)
            else:
                row[f"impact_{h}"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)


horizons = [500, 1000, 2000, 5000, 10000]

for trader, product in [("Mark 38", "HYDROGEL_PACK"), ("Mark 38", "VEV_4000"),
                         ("Mark 55", "VELVETFRUIT_EXTRACT"), ("Mark 14", "HYDROGEL_PACK")]:
    df_impact = price_impact(trader, product, horizons)
    impact_cols = [f"impact_{h}" for h in horizons]
    means = df_impact[impact_cols].mean()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(len(horizons)), means.values,
           color=["#55A868" if v > 0 else "#C44E52" for v in means.values], alpha=0.8)
    ax.set_xticks(range(len(horizons)))
    ax.set_xticklabels([f"+{h} ticks" for h in horizons])
    ax.axhline(0, color="black", lw=0.7)
    ax.set_title(f"{trader} | {product} — Avg Price Impact (signed in trade direction)", fontsize=10)
    ax.set_ylabel("Mid price change (signed)")
    ax.grid(True, alpha=0.3, axis="y")
    for i, v in enumerate(means.values):
        ax.text(i, v + np.sign(v) * 0.1, f"{v:.2f}", ha="center", fontsize=8)
    plt.tight_layout()
    plt.show()

## 8 — Inventory Reversal Levels

Do traders reverse direction at specific inventory thresholds?

In [ ]:
def reversal_analysis(trader, product):
    """Find inventory level at which trader switches from buying to selling or vice versa."""
    t = act[trader][act[trader]["symbol"] == product].copy().sort_values("global_ts").reset_index(drop=True)
    if len(t) < 4:
        return None
    t["inventory"] = t["signed_qty"].cumsum()
    t["next_side"]  = t["side"].shift(-1)
    reversals = t[t["side"] != t["next_side"]].dropna()
    return reversals[["global_ts", "side", "next_side", "inventory", "price"]]


for trader, product in [("Mark 14", "HYDROGEL_PACK"), ("Mark 38", "HYDROGEL_PACK"),
                         ("Mark 14", "VELVETFRUIT_EXTRACT"), ("Mark 55", "VELVETFRUIT_EXTRACT"),
                         ("Mark 14", "VEV_4000"), ("Mark 38", "VEV_4000")]:
    rev = reversal_analysis(trader, product)
    if rev is None or len(rev) == 0:
        continue
    buy_to_sell = rev[rev["next_side"] == "sell"]["inventory"]
    sell_to_buy = rev[rev["next_side"] == "buy"]["inventory"]
    print(f"{trader} | {product}")
    print(f"  Switches buy→sell at inventory:  mean={buy_to_sell.mean():.1f}  min={buy_to_sell.min():.0f}  max={buy_to_sell.max():.0f}  (n={len(buy_to_sell)})")
    print(f"  Switches sell→buy at inventory:  mean={sell_to_buy.mean():.1f}  min={sell_to_buy.min():.0f}  max={sell_to_buy.max():.0f}  (n={len(sell_to_buy)})")
    print()

In [ ]:
# Visualise inventory + reversal points
for trader, product in [("Mark 14", "HYDROGEL_PACK"), ("Mark 38", "HYDROGEL_PACK"),
                         ("Mark 14", "VELVETFRUIT_EXTRACT"), ("Mark 55", "VELVETFRUIT_EXTRACT")]:
    t = act[trader][act[trader]["symbol"] == product].copy().sort_values("global_ts").reset_index(drop=True)
    t["inventory"] = t["signed_qty"].cumsum()
    t["next_side"]  = t["side"].shift(-1)
    rev = t[t["side"] != t["next_side"]].dropna()

    fig, ax = plt.subplots(figsize=(16, 4))
    ax.step(t["global_ts"], t["inventory"], color="#4C72B0", lw=1.0, where="post", label="Inventory")
    b2s = rev[rev["next_side"] == "sell"]
    s2b = rev[rev["next_side"] == "buy"]
    ax.scatter(b2s["global_ts"], b2s["inventory"], color="#C44E52", s=30, zorder=5, label="buy→sell reversal")
    ax.scatter(s2b["global_ts"], s2b["inventory"], color="#55A868", s=30, zorder=5, label="sell→buy reversal")
    ax.axhline(0, color="black", lw=0.7, ls="--")
    for i in range(1, len(days)):
        ax.axvline(i * ticks_per_day, color="grey", ls="--", lw=0.6, alpha=0.5)
    ax.set_title(f"{trader} | {product} — Inventory with Direction Reversals", fontsize=10)
    ax.set_ylabel("Net position")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 9 — Mark 55 Intermediary Flow: Mark 01 → Mark 55 → Mark 14

In [ ]:
"""Mark 55 sells to Mark 01 (pure buyer), then buys from Mark 14 to flatten.
   We check: after Mark 55 sells to Mark 01, how quickly do they buy from Mark 14?"""

vex_55 = act["Mark 55"][act["Mark 55"]["symbol"] == "VELVETFRUIT_EXTRACT"].copy().sort_values("global_ts").reset_index(drop=True)
vex_55["cum_inv"] = vex_55["signed_qty"].cumsum()

# tag the counterparty
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

# inventory
ax1.step(vex_55["global_ts"], vex_55["cum_inv"], color="#4C72B0", lw=1.0, where="post", label="Inventory")

# scatter trades, coloured by counterparty
cp_colors = {"Mark 01": "#E377C2", "Mark 14": "#17BECF", "Mark 22": "#BCBD22",
             "Mark 49": "#9467BD", "Mark 67": "#8C564B"}
for cp, color in cp_colors.items():
    sub = vex_55[vex_55["counterparty"] == cp]
    buy_sub  = sub[sub["side"] == "buy"]
    sell_sub = sub[sub["side"] == "sell"]
    ax2.scatter(buy_sub["global_ts"],  buy_sub["price"],  color=color, marker="^", s=15, alpha=0.7, label=f"buy ← {cp}")
    ax2.scatter(sell_sub["global_ts"], sell_sub["price"], color=color, marker="v", s=15, alpha=0.7, label=f"sell → {cp}")

mid_vex = prices[prices["product"] == "VELVETFRUIT_EXTRACT"][["global_ts","mid_price"]]
ax2.plot(mid_vex["global_ts"], mid_vex["mid_price"], color="grey", lw=0.5, alpha=0.5, label="Mid")

for ax in (ax1, ax2):
    ax.axhline(0 if ax == ax1 else ax2.get_ylim()[0], color="black", lw=0.5)
    for i in range(1, len(days)):
        ax.axvline(i * ticks_per_day, color="grey", ls="--", lw=0.6, alpha=0.5)
    ax.grid(True, alpha=0.3)

ax1.set_title("Mark 55 VELVETFRUIT — Inventory", fontsize=10)
ax1.set_ylabel("Net position")
ax1.legend(fontsize=8)
ax2.set_title("Mark 55 — Trade Prices by Counterparty (▲ buy, ▼ sell)", fontsize=10)
ax2.set_ylabel("Price")
ax2.set_xlabel("Global timestamp")
ax2.legend(fontsize=7, ncol=3)
plt.tight_layout()
plt.show()

# average prices paid/received per counterparty
print("Mark 55 — average trade price by counterparty and side:")
print(vex_55.groupby(["counterparty","side"])["price"].agg(["mean","std","count"]).round(2))

## 10 — Summary of Patterns

In [ ]:
print("=" * 72)
print("MARKET STRUCTURE PATTERNS — MARK 14 / 38 / 55")
print("=" * 72)
print("""
MARK 14 — Pure Market Maker
  • Posts both bid and ask across HYDROGEL_PACK, VELVETFRUIT_EXTRACT, VEV_4000.
  • Consistently buys below mid, sells above mid → captures the spread.
  • Spread: ~13 ticks (HP), ~5 ticks (VEX), ~19 ticks (VEV_4000).
  • Inventory mean-reverts within each day — does not carry overnight risk.
  • No exclusive counterparty in VEX: trades with Mark 55 and Mark 22.
  • In VEV_4000 and HP: almost exclusively counterparty is Mark 38.

MARK 38 — Directional Taker (vs Mark 14 in HP + VEV_4000)
  • Always pays Mark 14's spread — buys at his ask, sells at his bid.
  • Negative captured spread (-13 HP, -19 VEV_4000) → only profitable
    if trades are informed and mid moves in their favour.
  • See Section 7 price impact chart: positive impact → Mark 38 IS informed.
  • Trading implication: when Mark 38 is buying HP or VEV_4000 heavily,
    price likely moving up. Fade or join depending on your position.

MARK 55 — Intermediary / Layered Market Maker in VEX
  • Buys from Mark 14 (paying spread ~5 ticks), then sells to Mark 01/67.
  • Also buys from Mark 01 sometimes (Mark 01 occasionally sells).
  • Average sell price to Mark 01 > average buy price from Mark 14.
    Mark 55 profits by capturing the spread between the two.
  • Inventory oscillates — short when selling to Mark 01, then flattens
    by buying from Mark 14.

TRADING IMPLICATIONS
  • Mark 14 defines the mid: his bid/ask bracket is the effective market.
    Your algo should quote inside his spread to capture order flow.
  • Mark 38 gives directional signal: net buying = bullish for HP/VEV_4000.
  • Mark 55's short-selling of VEX to Mark 01 compresses VEX bid/ask;
    when Mark 01 is very active, VEX might be slightly overbought.
  • Reversal levels (Section 8) show rough inventory limits — if you can
    push a MM beyond their comfort zone, they will switch sides.
""")